## Imports and Configs

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

DATA_PATH = "/home/zaphod/Documents/SampleLogs/spam_ham_dataset.csv"
assert os.path.exists(DATA_PATH), f"Dataset not found: {DATA_PATH}"

print("Environment ready")


Environment ready


## Load & Inspect Dataset

In [2]:
# load a dataframe

# Load dataset (UTF-8 fallback to Latin-1)
encodings = ["utf-8", "latin-1", "cp1252"]
df = None
for enc in encodings:
    try:
        df = pd.read_csv(DATA_PATH, encoding=enc)
        break
    except:
        pass

if df is None:
    raise ValueError("Unable to read dataset with common encodings.")

print(df.head())
print("\nShape is rows/columns")
print("Shape:", df.shape)
# column names
print("\nColumns:", df.columns.tolist())


   Unnamed: 0 label                                               text  \
0         605   ham  Subject: enron methanol ; meter # : 988291\r\n...   
1        2349   ham  Subject: hpl nom for january 9 , 2001\r\n( see...   
2        3624   ham  Subject: neon retreat\r\nho ho ho , we ' re ar...   
3        4685  spam  Subject: photoshop , windows , office . cheap ...   
4        2030   ham  Subject: re : indian springs\r\nthis deal is t...   

   label_num  
0          0  
1          0  
2          0  
3          1  
4          0  

Shape is rows/columns
Shape: (5171, 4)

Columns: ['Unnamed: 0', 'label', 'text', 'label_num']


## Prepare Clean Data & Split Train/Test

In [3]:
# define column names
LABEL_COL = "label"
TEXT_COL = "text"

# normalize it
# validates strings, removes whitespace
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()

# split the data for training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL],
    df[LABEL_COL],
    test_size=0.2,
    random_state=42,
    stratify=df[LABEL_COL]
)

# print data set sizes
print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 4136
Test size: 1035


## Train Baseline Classifier

In [4]:
# create a scikit-learn pipeline
clf = Pipeline([
    # Term Frequency-Inverse Document Frequency
    # Use single words (unigrams) and word pairs (bigrams)
    # ignore extremely rare terms that appear only once to reduce noise
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2)),
    # use log regression from scikit-learn
    # limit of how many times it to refine solution
    # run on all cores
    ("logreg", LogisticRegression(max_iter=200, n_jobs=-1))
])

# apply TF-IDF transformations
# train the logreg model on the transformed data
# store it to the model
clf.fit(X_train, y_train)
print("Model trained")


Model trained


## Evaluate and Save Model

In [5]:
# Use the trained pipeline and the test data
pred = clf.predict(X_test)

# accuracy, higher the better
print("Accuracy:", accuracy_score(y_test, pred))
print("\nClassification Report:\n")
print("Precision - When model says \"spam\", how often is it correct?")
print("Recall - How many real spam messages did it catch?")
print("F1-score - Balanced score between precision & recall\n")

print(classification_report(y_test, pred))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, pred))
print("Correct: Ham predicted as Ham - True Negatives (TN)")
print("Incorrect: Ham predicted as Spam	- False Positives (FP)")
print("Incorrect: Spam predicted as Ham	- False Negatives (FN)")
print("Correct: Spam predicted as Spam - True Positives (TP)")

MODEL_PATH = "spam_classifier.pkl"
joblib.dump(clf, MODEL_PATH)
print(f"\nModel saved to: {MODEL_PATH}")


Accuracy: 0.9874396135265701

Classification Report:

Precision - When model says "spam", how often is it correct?
Recall - How many real spam messages did it catch?
F1-score - Balanced score between precision & recall

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       735
        spam       0.98      0.97      0.98       300

    accuracy                           0.99      1035
   macro avg       0.99      0.98      0.98      1035
weighted avg       0.99      0.99      0.99      1035


Confusion Matrix:

[[730   5]
 [  8 292]]
Correct: Ham predicted as Ham - True Negatives (TN)
Incorrect: Ham predicted as Spam	- False Positives (FP)
Incorrect: Spam predicted as Ham	- False Negatives (FN)
Correct: Spam predicted as Spam - True Positives (TP)

Model saved to: spam_classifier.pkl


## Create Embeddings for the Training Messages

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print("Embedding model loaded")

# Convert training text into embeddings
train_embeddings = embed_model.encode(
    X_train.tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

# Keep metadata: store original text + labels
train_texts = X_train.tolist()
train_labels = y_train.tolist()

print("Embeddings shape:", train_embeddings.shape)


Embedding model loaded


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Embeddings shape: (4136, 384)


## Build Vector Store (FAISS) + attach metadata

In [7]:
import faiss

# Dimension of embeddings
d = train_embeddings.shape[1]

# Create index (flat L2 distance)
index = faiss.IndexFlatL2(d)

# Add embeddings to index
index.add(train_embeddings)

print("FAISS index built")
print("Indexed vectors:", index.ntotal)


FAISS index built
Indexed vectors: 4136


## retrieval function  
  - Feed in a new message
  - Embed it
  - Query the FAISS index
  - Return the most similar historical examples, including labels + text

In [8]:
def retrieve_similar_examples(query_text, k=5):
    """
    Given an input message, return the top-k most similar
    messages from the training set along with labels and distances.
    """
    # Encode the query message into an embedding vector
    query_embedding = embed_model.encode([query_text], convert_to_numpy=True)
    
    # Search the FAISS index
    distances, indices = index.search(query_embedding, k)

    # Gather results
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            "text": train_texts[idx],
            "label": train_labels[idx],
            "distance": float(dist)
        })
    
    return results


### simple test

In [9]:
sample_query = "You won a prize! Claim your reward now!"
neighbors = retrieve_similar_examples(sample_query, k=5)

for i, item in enumerate(neighbors, start=1):
    print(f"\nResult #{i}")
    print("Label:", item["label"])
    print("Distance:", round(item["distance"], 4))
    print("Text:", item["text"])



Result #1
Label: spam
Distance: 0.9608
Text: Subject: congratulations
from : the lottery coordinator ,
international promotions / prize award department
dear sir / madam ,
results for third category draws
royal stakes south africa wishes to inform you of the
results of it ' s third category draws held on the 3 rd
april 2004 . we are happy to officially inform you that
you have emerged a winner under our third category
draws , which is part of our promotional draws .
participants were selected through a computer ballot
system drawn from 40 , 000 names / email addresses of
individuals and companies from africa , america , asia ,
canada , europe , middle east , and oceania as part of
our international promotions program .
you / your company , attached to ticket number
28 - 04 - 1376 , with serial number 20 - 431 drew the lucky
numbers 12 , 18 , 21 , 32 , 41 , 47 ( 31 ) , and consequently
won in the third category .
you have therefore been awarded a lump sum pay out of
us $ 2 , 500 , 000 

## LLM-Powered Explanation Function

In [10]:
import json
import subprocess

OLLAMA_MODEL = "llama3.1:8b"

def generate_explanation(input_text, prediction, evidence, max_evidence=3):
    """
    Ask the local LLM to explain the reasoning behind the spam classification,
    using retrieved similar examples as supporting points.
    """

    # Format retrieved examples for the prompt
    evid_text = "\n\n".join([
        f"Example {i+1} (label={ev['label']}): {ev['text']}"
        for i, ev in enumerate(evidence[:max_evidence])
    ])

    prompt = f"""
You are an expert email spam analyzer. Classify the message and explain your reasoning
using the similar examples provided. Be concise.

Message:
\"\"\"{input_text}\"\"\"

Model Prediction: {prediction}

Similar Messages:
{evid_text}

Explain why this message is labeled '{prediction}' and include a short 'reason code: ___'. 
"""

    # Call Ollama and get JSON response
    process = subprocess.Popen(
        ["ollama", "run", OLLAMA_MODEL],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    output, err = process.communicate(prompt)

    if err:
        print("LLM error:", err)

    return output.strip()


### test LLM

In [11]:
test_message = "Your account has been suspended! Click here to verify your password."
pred = clf.predict([test_message])[0]
neighbors = retrieve_similar_examples(test_message, k=5)

explanation = generate_explanation(test_message, pred, neighbors)
print(explanation)


LLM error: ⠋ ⠹ ⠹ ⠸ 
**Classification:** Spam
**Reason Code:** URGE-VERIF-LINK (Urgent verification link without validation)

This message triggers my spam detector due to the following characteristics:

1. **Urgency**: The message creates a sense of urgency by stating that the account has been suspended and needs immediate verification.
2. **Verification link**: A clickable link is provided without any additional information or validation steps, which is suspicious.
3. **Lack of personalization**: There's no indication that the email was addressed to a specific individual or tailored to their account details.

These red flags are consistent with the patterns seen in Examples 1 and 2, which were also labeled as spam.


## Classify and Explain

In [12]:
import numpy as np

def classify_and_explain(message, k=5):
    """
    Full RAG-enhanced spam classification pipeline:
    Predict spam / ham using trained classifier
    Retrieve similar examples for evidence
    Generate LLM explanation
    Print a clean, user-friendly result
    """

    print("=" * 60)
    print("Incoming Message:")
    print(message)
    print("=" * 60)

    # Classifier prediction + confidence
    pred = clf.predict([message])[0]
    probs = clf.predict_proba([message])[0]
    label_index = list(clf.classes_).index(pred)
    confidence = probs[label_index]

    print(f" Prediction: {pred.upper()}")
    print(f" Confidence: {confidence:.4f}")

    # Retrieve neighbors
    similar_examples = retrieve_similar_examples(message, k=k)

    #  LLM explanation
    explanation = generate_explanation(message, pred, similar_examples)

    print("\n LLM Explanation:")
    print(explanation)

    print("\n Similar Examples Retrieved:")
    for i, item in enumerate(similar_examples, start=1):
        print(f"\nExample #{i}")
        print(f"Label: {item['label']}")
        print(f"Distance: {item['distance']:.4f}")
        print(f"Text: {item['text']}")

    print("=" * 60)
    return {
        "message": message,
        "prediction": pred,
        "confidence": confidence,
        "explanation": explanation,
        "neighbors": similar_examples,
    }


### test cases

In [13]:
classify_and_explain("Your account has been suspended! Click to restore access.")

Incoming Message:
Your account has been suspended! Click to restore access.
 Prediction: SPAM
 Confidence: 0.5495
LLM error: ⠋ ⠹ ⠸ ⠸ ⠴ 

 LLM Explanation:
I would classify this message as spam.

Reason Code: URG-UNSOLICITED-REQUEST-FOR-ACTION

Explanation:

* The message creates a sense of urgency by stating that the account has been suspended.
* There is no personalized content or information to suggest that the recipient actually has an account with the company.
* The message is trying to trick the user into clicking on a link without verifying its authenticity, which is a common tactic used in phishing scams.

This type of message is similar to Examples 1 and 2, both labeled as spam.

 Similar Examples Retrieved:

Example #1
Label: spam
Distance: 1.1467
Text: Subject: note ! citibank account suspend in process
dear customer :
recently there have been a large number of cyber attacks pointing our database servers . in order to safeguard your account , we require you to sign on immedia

{'message': 'Your account has been suspended! Click to restore access.',
 'prediction': 'spam',
 'confidence': 0.5494643218507408,
 'explanation': 'I would classify this message as spam.\n\nReason Code: URG-UNSOLICITED-REQUEST-FOR-ACTION\n\nExplanation:\n\n* The message creates a sense of urgency by stating that the account has been suspended.\n* There is no personalized content or information to suggest that the recipient actually has an account with the company.\n* The message is trying to trick the user into clicking on a link without verifying its authenticity, which is a common tactic used in phishing scams.\n\nThis type of message is similar to Examples 1 and 2, both labeled as spam.',
 'neighbors': [{'text': 'Subject: note ! citibank account suspend in process\r\ndear customer :\r\nrecently there have been a large number of cyber attacks pointing our database servers . in order to safeguard your account , we require you to sign on immediately .\r\nthis personal check is requeste

In [14]:
classify_and_explain("Hey, are we still on for lunch tomorrow?")

Incoming Message:
Hey, are we still on for lunch tomorrow?
 Prediction: HAM
 Confidence: 0.6665
LLM error: ⠋ ⠹ 

 LLM Explanation:
The message "Hey, are we still on for lunch tomorrow?" is correctly labeled as 'ham'.

**Reason Code:** __SOCIAL_INVITATION__

This message meets the criteria of a 'social invitation', which is a common type of ham (non-spam) email. The language and tone used are informal and friendly, indicating that it's an invitation from someone you know to meet up for lunch.

 Similar Examples Retrieved:

Example #1
Label: ham
Distance: 1.2055
Text: Subject: lunch
treebeard ' s
?
i am hooked . name the time and place to meet . . . . . . . . . . . . . . . . . . . . . .
?
tell ami good luck for me .
?
who would have thunk ou would be back so quickly . now we have to contend
with them , k - state , and the conference championship ( if we survive ) . i don ' t
see the big 12 being # 1 if we don ' t lighten up on each other .
?
you want a chuckle - - - - - my daughter is a 

{'message': 'Hey, are we still on for lunch tomorrow?',
 'prediction': 'ham',
 'confidence': 0.6664802177213679,
 'explanation': 'The message "Hey, are we still on for lunch tomorrow?" is correctly labeled as \'ham\'.\n\n**Reason Code:** __SOCIAL_INVITATION__\n\nThis message meets the criteria of a \'social invitation\', which is a common type of ham (non-spam) email. The language and tone used are informal and friendly, indicating that it\'s an invitation from someone you know to meet up for lunch.',
 'neighbors': [{'text': "Subject: lunch\r\ntreebeard ' s\r\n?\r\ni am hooked . name the time and place to meet . . . . . . . . . . . . . . . . . . . . . .\r\n?\r\ntell ami good luck for me .\r\n?\r\nwho would have thunk ou would be back so quickly . now we have to contend\r\nwith them , k - state , and the conference championship ( if we survive ) . i don ' t\r\nsee the big 12 being # 1 if we don ' t lighten up on each other .\r\n?\r\nyou want a chuckle - - - - - my daughter is a big nu f